# C.4 找得到也選得中嗎？


2+2的候選是3、3、4。正解已經在集合，多數決仍會選3。要分開問「有沒有找到」與「怎樣交出正解」，否則更多候選也可能只讓同一錯解得更多票。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
# @title 本節圖解
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/rewrite-C-candidate-choice.svg")))

In [ ]:
from collections import Counter

candidates = [3, 3, 4]
majority = Counter(candidates).most_common(1)[0][0]
verified = next((x for x in candidates if x == 2 + 2), None)
print("集合包含正解", 4 in candidates)
print("多數決", majority, "答對", majority == 4)
print("驗證器", verified, "答對", verified == 4)

集合含正解True；Counter數兩票3、一票4，選3、False。另一條用2+2驗算，next取第一個通過的4、True；全部不通過時None，沒有從空的正解集合補一個答案。

多數決看出現次數，不懂等式。驗算路線則有可直接計算的真值，是這個窄任務的強條件，不假定未知事實也有這樣的驗證器。開放書店地址不能靠預埋青街8號來證明一般選擇能力。

舊短答題 `(3+3)+0` 曾抽到 `[5,7,5,6]`，集合有6卻投票選5。這個對照用同一批候選，沒有把不同採樣混比。可以分別記完整題目最終正確率，以及「確有正解的集合」裏選對比例；後者分母更小，不能都叫同一個成功率。

<details>
<summary>補充：舊多數決、平票與精確驗算結果</summary>

實際短答候選也重現了選錯：`(3+3)+0=?`在k=4時生成`[5,7,5,6]`，真值6已在集合，多數決仍交5而答錯。精確算術驗證器才選到6。下表都使用各k保存的同一批候選，各欄的分母是完整24題：

| k | 短答多數決 | 短答驗證器 | 步驟多數決 | 步驟驗證器 |
| --- | --- | --- | --- | --- |
| 1 | 1/24 | 1/24 | 11/24 | 11/24 |
| 2 | 1/24 | 1/24 | 12/24 | 13/24 |
| 4 | 1/24 | 2/24 | 12/24 | 13/24 |
| 8 | 1/24 | 1/24 | 12/24 | 14/24 |

這份多數決取最多次的可解析最後整數，平票則取先出現者；不是要求超過一半票數。步驟k=2的`(5+0)+3=?`先生成10，再生成8，平票取10而錯，驗證器檢查等式後選8。步驟k=8若只看已有正解的14題，多數決選對12/14；它與全體12/24回答的是不同問題。

本輪精確驗證器選第一份全部檢查通過的候選，沒有教師模型幫忙挑答案；它能直接計算這個窄任務的真值，因此選擇成績和oracle coverage相同。這個很強的條件不能延伸到未知事實與開放論證。多份候選同意，或可見等式都正確，也不足以證明內部chain of thought的忠實性。

</details>
